# Notebook 03: Generación de Pseudo-Labels

## Objetivo
Generar pseudo-labels (JSON estructurado) para cada chunk de transcripción usando **Ollama local** (qwen2.5:14b), con fallback a heurísticas si el LLM no está disponible.

## Pipeline
1. Cargar `raw_train.jsonl` y `raw_val.jsonl` (generados por el script `add_session_to_raw_train.py`)
2. Para cada ejemplo, generar un JSON con: topic, explanation, bullets, example, formula, keywords, timestamps_ms, source
3. Validar y sanitizar cada output (schema + limpieza de metadatos)
4. Guardar `final_train.jsonl` y `final_val.jsonl`

## Inputs
- `data/raw_train.jsonl`: dataset sin labels
- `data/raw_val.jsonl`: dataset de validación sin labels

## Outputs
- `data/final_train.jsonl`: dataset completo con pseudo-labels
- `data/final_val.jsonl`: validation set completo
- `outputs/pseudolabel_stats.json`: estadísticas de calidad

---
## Setup: Instalación de Dependencias

In [1]:
import sys
import os
from copy import deepcopy
IN_COLAB = 'COLAB_GPU' in os.environ
try:
    IN_COLAB = IN_COLAB or ('google.colab' in str(get_ipython()))
except NameError:
    pass
if IN_COLAB:
    os.system("pip install -q requests tqdm")
import json
import re
import requests
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from tqdm.auto import tqdm
import time
print(" Dependencias listas")
print(f" Entorno: {'Google Colab' if IN_COLAB else 'Local (VS Code / Jupyter)'}")

 Dependencias listas
 Entorno: Local (VS Code / Jupyter)


---
## Configuración

In [2]:
# Paths
if IN_COLAB:
    PROJECT_ROOT = Path("/content")
else:
    cwd = Path.cwd()
    # Proyecto_Final_Alejandro/notebooks/ → subir 2 niveles a video/
    if cwd.name == "notebooks" and cwd.parent.name == "Proyecto_Final_Alejandro":
        PROJECT_ROOT = cwd.parent.parent
    elif cwd.name in ("notebooks_colab", "Proyecto_Final_Alejandro"):
        PROJECT_ROOT = cwd.parent
    else:
        PROJECT_ROOT = cwd
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True, parents=True)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
RAW_TRAIN_PATH = DATA_DIR / "raw_train.jsonl"
RAW_VAL_PATH = DATA_DIR / "raw_val.jsonl"
FINAL_TRAIN_PATH = DATA_DIR / "final_train.jsonl"
FINAL_VAL_PATH = DATA_DIR / "final_val.jsonl"
CHECKPOINT_TRAIN = DATA_DIR / "checkpoint_train.jsonl"
CHECKPOINT_VAL = DATA_DIR / "checkpoint_val.jsonl"
STATS_PATH = OUTPUT_DIR / "pseudolabel_stats.json"

# Método de generación
GENERATION_METHOD = "llama_serve"  # "llama_serve" | "heuristic"

# Config Ollama
LLAMA_BASE_URL = "http://localhost:11434"
LLAMA_MODEL = "qwen2.5:14b"

# DRY_RUN: True para test rápido, False para procesar todo
# Dataset actual: LLM Sessions 38-43 (~555 train, ~125 val)
DRY_RUN = False
DRY_RUN_N = 10
DRY_RUN_SHOW = 3
DRY_SAMPLE = "random"  # "random" | "first" | "shortest" | "longest"

# Checkpoint
CHECKPOINT_EVERY = 25

# Retries
MAX_RETRIES = 2

print(f"Project root: {PROJECT_ROOT}")
print(f"Metodo    : {GENERATION_METHOD}")
print(f"Modelo    : {LLAMA_MODEL} | URL: {LLAMA_BASE_URL}")
print(f"DRY_RUN   : {DRY_RUN}" + (f" ({DRY_RUN_N} ejemplos, muestra={DRY_SAMPLE})" if DRY_RUN else " (PRODUCCION - todos los ejemplos)"))
print(f"Checkpoint: cada {CHECKPOINT_EVERY} ejemplos")
print(f"Retries   : {MAX_RETRIES}")

Project root: /Users/alexmartin/Documents/video
Metodo    : llama_serve
Modelo    : qwen2.5:14b | URL: http://localhost:11434
DRY_RUN   : False (PRODUCCION - todos los ejemplos)
Checkpoint: cada 25 ejemplos
Retries   : 2


---
## Cargar Dataset Raw

In [3]:
def load_jsonl(filepath: Path) -> List[Dict]:
    """Carga archivo JSONL."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

def save_jsonl(data: List[Dict], filepath: Path):
    """Guarda lista de dicts como JSONL."""
    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

train_data = load_jsonl(RAW_TRAIN_PATH)
val_data = load_jsonl(RAW_VAL_PATH)

# Mostrar sesiones en el dataset
train_sessions = sorted(set(ex['meta']['session_id'] for ex in train_data))
val_sessions = sorted(set(ex['meta']['session_id'] for ex in val_data))

print(f"Datasets cargados:")
print(f"  Train: {len(train_data)} ejemplos <- sesiones: {train_sessions}")
print(f"  Val  : {len(val_data)} ejemplos <- sesiones: {val_sessions}")

Datasets cargados:
  Train: 762 ejemplos <- sesiones: ['llm_38', 'llm_39', 'llm_40', 'llm_41', 'llm_42', 'llm_43', 'llm_48', 'llm_50']
  Val  : 125 ejemplos <- sesiones: ['llm_43']


---
## Generación con Ollama (Local)

Usa Ollama con `qwen2.5:14b` para generar pseudo-labels en formato JSON estructurado.
Requiere Ollama corriendo:
```bash
ollama serve
```

In [4]:
def ping_ollama() -> bool:
    try:
        r = requests.get(f"{LLAMA_BASE_URL}/api/tags", timeout=5)
        r.raise_for_status()
        models = [m["name"] for m in r.json().get("models", [])]
        print(f" Ollama OK — modelos: {models}")
        if not any(LLAMA_MODEL in m for m in models):
            print(f" Modelo '{LLAMA_MODEL}' no encontrado. Descárgalo: ollama pull {LLAMA_MODEL}")
        return True
    except Exception as e:
        print(f" Ollama no responde: {e}")
        return False
def build_clean_messages(messages: List[Dict]) -> List[Dict]:
    """Reconstruye el user content como texto limpio (sin claves de metadatos en el cuerpo)."""
    clean = []
    for msg in messages:
        if msg["role"] == "user":
            c = msg["content"]
            # Extraer campos del formato del NB01
            m_slide = re.search(r'^Slide:\s*(.+?)(?:\n|$)', c, re.MULTILINE)
            m_subs = re.search(r'^Subtítulos:\s*(.+?)(?:\n|$)', c, re.MULTILINE)
            m_audio = re.search(r'Transcripción:\s*(.+?)(?:\n---|\Z)', c, re.DOTALL)
            m_id = re.search(r'^slide_id:\s*(\S+)', c, re.MULTILINE)
            m_ts = re.search(r'^timestamps_ms:\s*(\[.+?\])', c, re.MULTILINE)
            parts = []
            if m_slide:
                parts.append(f"Slide: {m_slide.group(1).strip()}")
            if m_subs:
                parts.append(f"Subtítulos: {m_subs.group(1).strip()}")
            if m_audio:
                parts.append(f"Transcripción: {m_audio.group(1).strip()[:1200]}")
            if m_id:
                parts.append(f"slide_id: {m_id.group(1).strip()}")
            if m_ts:
                parts.append(f"timestamps_ms: {m_ts.group(1).strip()}")
            clean.append({"role": "user", "content": "\n".join(parts) if parts else c})
        elif msg["role"] == "assistant" and not msg["content"].strip():
            continue
        else:
            clean.append(msg)
    return clean
def _call_ollama(messages: List[Dict]) -> str:
    """Llamada cruda a la API de Ollama."""
    payload = {
        "model": LLAMA_MODEL,
        "messages": messages,
        "stream": False,
        "options": {"temperature": 0.3, "num_predict": 700},
    }
    resp = requests.post(f"{LLAMA_BASE_URL}/api/chat", json=payload, timeout=180)
    resp.raise_for_status()
    return resp.json()["message"]["content"]
def generate_with_ollama(messages: List[Dict]) -> str:
    """
    Genera con Ollama. Si el output no es JSON válido, reintenta MAX_RETRIES veces
    enviando el error como feedback ("Fix your JSON").
    """
    clean_msgs = build_clean_messages(messages)
    content = _call_ollama(clean_msgs)
    # Intento 0: evaluar si ya es válido
    is_valid, _ = validate_json_output(content)
    if is_valid:
        return content
    # Reintentos con feedback
    retry_msgs = list(clean_msgs)
    for attempt in range(1, MAX_RETRIES + 1):
        retry_msgs.append({"role": "assistant", "content": content})
        retry_msgs.append({
            "role": "user",
            "content": (
                "Tu respuesta anterior no es JSON válido o no cumple el schema.\n"
                "Reglas que debes cumplir:\n"
                "- bullets: EXACTAMENTE 5 strings\n"
                "- timestamps_ms: [start_ms, end_ms] con start_ms < end_ms\n"
                "- source: {\"slide_id\": \"...\", \"evidence\": \"...\"}\n"
                "- formula: string o null\n"
                "- Cero texto fuera del JSON\n\n"
                "Devuelve SOLO el JSON corregido."
            )
        })
        content = _call_ollama(retry_msgs)
        is_valid, _ = validate_json_output(content)
        if is_valid:
            return content
    # Agotados los reintentos: devolver crudo (se marcará como error en el loop)
    return content
server_ok = ping_ollama()

 Ollama OK — modelos: ['qwen2.5:7b', 'qwen2.5:14b']


---
## Heurísticas (Fallback sin LLM)

Si Ollama no está disponible o falla, se generan pseudo-labels con heurísticas basadas en el contenido de la transcripción.

In [5]:
# Constantes lingüísticas
STOPWORDS_ES = {
    "que", "de", "el", "la", "los", "las", "un", "una", "y", "en", "a",
    "por", "con", "se", "es", "son", "para", "lo", "le", "si", "no", "su",
    "como", "del", "al", "más", "pero", "esto", "este", "esta", "ese", "eso",
    "también", "ya", "muy", "hay", "pues", "vale", "bien", "aquí", "cuando",
    "porque", "sobre", "entre", "desde", "hasta", "vamos", "voy", "ver",
    "todo", "todos", "cada", "así", "entonces", "tenemos", "podemos", "puede",
    "tienen", "tiene", "hacer", "decir", "creo", "digo", "tipo", "algo",
    "igual", "claro", "bueno", "ahora", "aqui", "ahi", "alli", "luego",
    "antes", "después", "mismo", "nuestra", "nuestro",
}
BAD_STARTERS = {"y", "a", "o", "e", "pero", "entonces", "vale", "pues",
    "si", "como", "aunque", "además", "también", "bueno"}
REQUIRED_FIELDS = {'topic', 'explanation', 'bullets', 'example',
    'formula', 'keywords', 'timestamps_ms', 'source'}
METADATA_RE = re.compile(
    r'\b(SLIDE_OCR|SUBTITLES_OCR|TEACHER_AUDIO|TIME_RANGE|SLIDE_ID|'
    r'frame_\d*|Genera los apuntes|genera los apuntes)\b|'
    r'\b\d+ms\b',
    re.IGNORECASE
)
JUNK_KEYWORDS = {'slide', 'frame', 'time', 'range', 'ocr', 'audio', 'teacher',
    'genera', 'apuntes', 'json', 'input', 'output', 'subtítulos',
    'transcripción'}
# extract_json
def extract_json(content: str) -> str:
    """Extrae el primer objeto JSON del string, ignora fences markdown."""
    content = re.sub(r'^```(?:json)?\s*\n?', '', content.strip(), flags=re.MULTILINE)
    content = re.sub(r'\n?```\s*$', '', content)
    m = re.search(r'\{[\s\S]*\}', content)
    return m.group(0) if m else content
# validate_json_output
def validate_json_output(content: str) -> Tuple[bool, Optional[Dict]]:
    """
    Schema: topic, explanation, bullets[5], example, formula,
    keywords, timestamps_ms[2], source{slide_id, evidence}
    """
    try:
        data = json.loads(extract_json(content))
    except Exception:
        return False, None
    if REQUIRED_FIELDS - set(data.keys()):
        return False, None
    bullets = data.get('bullets', [])
    if not isinstance(bullets, list) or len(bullets) != 5:
        return False, None
    if not all(isinstance(b, str) and len(b.strip()) > 5 for b in bullets):
        return False, None
    ts = data.get('timestamps_ms', [])
    if not (isinstance(ts, list) and len(ts) == 2):
        return False, None
    try:
        if int(ts[0]) >= int(ts[1]):
            return False, None
    except (TypeError, ValueError):
        return False, None
    src = data.get('source', {})
    if not isinstance(src, dict) or 'slide_id' not in src or 'evidence' not in src:
        return False, None
    formula = data.get('formula')
    if formula is not None and not isinstance(formula, str):
        return False, None
    if not isinstance(data.get('keywords', []), list):
        return False, None
    return True, data
# sanitize_output
def sanitize_output(data: Dict, user_content: str = "") -> Tuple[Dict, bool]:
    """Limpia contaminación de metadatos. Rellena bullets hasta exactamente 5."""
    data = dict(data)
    # bullets: eliminar contaminados, rellenar hasta 5
    clean = []
    for b in data.get("bullets", []):
        if METADATA_RE.search(b):
            continue
        words = b.split()
        if len(words) > 12:
            b = " ".join(words[:12]) + "."
        if len(b.strip()) > 10:
            clean.append(b)
    while len(clean) < 5:
        clean.append("Consulta la grabación para ampliar este punto.")
    data["bullets"] = clean[:5]
    # topic
    topic = data.get("topic") or ""
    if not isinstance(topic, str) or METADATA_RE.search(topic):
        m = re.search(r'^Slide:\s*(.+)', user_content, re.MULTILINE)
        data["topic"] = re.sub(r'\s*[-–]\s*.*', '', m.group(1).strip())[:60] if m \
            else " ".join(data["bullets"][0].split()[:4]).rstrip(".")
    # explanation
    explanation = data.get("explanation") or ""
    if not isinstance(explanation, str) or METADATA_RE.search(explanation):
        data["explanation"] = data["bullets"][0]
    # example
    example = data.get("example") or ""
    if not isinstance(example, str) or METADATA_RE.search(example):
        data["example"] = "Ver la grabación para ejemplos concretos."
    # keywords
    keywords = data.get("keywords") or []
    if isinstance(keywords, list):
        data["keywords"] = [
            kw for kw in keywords
            if isinstance(kw, str) and kw.lower() not in JUNK_KEYWORDS and not METADATA_RE.search(kw)
        ][:6] or ["concepto", "clase"]
    else:
        data["keywords"] = ["concepto", "clase"]
    # formula
    if data.get("formula") is not None and not isinstance(data["formula"], str):
        data["formula"] = None
    # source.evidence <= 20 palabras
    src = data.get("source", {})
    if isinstance(src, dict):
        src["evidence"] = " ".join(str(src.get("evidence", "")).split()[:20])
        data["source"] = src
    return data, False
# extract_transcript
def extract_transcript(user_content: str) -> str:
    m = re.search(r'Transcripción:\s*(.+?)(?:\n---|\Z)', user_content, re.DOTALL)
    if m:
        return m.group(1).strip()
    m = re.search(r'TEACHER_AUDIO:\s*(.+?)(?:\nTIME_RANGE:|\nSLIDE_ID:|$)', user_content, re.DOTALL)
    if m:
        return m.group(1).strip()
    return user_content[:800]
# Heurística
def content_density(s: str) -> float:
    words = s.lower().split()
    return len([w for w in words if w not in STOPWORDS_ES and len(w) > 3]) / len(words) if words else 0.0
def clean_sentence(s: str) -> str:
    s = re.sub(r'\b(vale|o sea|pues|eh|este|bueno|entonces|digamos|tipo que)\b', '', s, flags=re.IGNORECASE)
    s = re.sub(r'\s+', ' ', s).strip().rstrip(',')
    return s[0].upper() + s[1:] if s else ""
def generate_with_heuristics(messages: List[Dict], slide_id: str, timestamp_ms: int) -> str:
    user_content = next(m["content"] for m in messages if m["role"] == "user")
    transcript = extract_transcript(user_content)
    m_slide = re.search(r'^Slide:\s*(.+)', user_content, re.MULTILINE)
    slide_ocr = m_slide.group(1).strip() if m_slide else ""
    m_ts = re.search(r'timestamps_ms:\s*\[(\d+),\s*(\d+)\]', user_content)
    ts_start = int(m_ts.group(1)) if m_ts else timestamp_ms
    ts_end = int(m_ts.group(2)) if m_ts else timestamp_ms + 60000
    raw = re.split(r'[.!?]\s+', transcript)
    valid = sorted(
        [clean_sentence(s) for s in raw
         if len(clean_sentence(s).split()) >= 5
         and clean_sentence(s).split()[0].lower() not in BAD_STARTERS
         and len(clean_sentence(s)) > 20],
        key=content_density, reverse=True
    )
    bullets = []
    for s in valid[:5]:
        b = " ".join(s.split()[:12])
        bullets.append(b if b.endswith(".") else b + ".")
    while len(bullets) < 5:
        bullets.append("Consulta la grabación para más detalles.")
    if slide_ocr:
        topic = re.sub(r'\s*[-–]\s*.*', '', slide_ocr).strip()[:50]
    elif valid:
        cands = [w for w in valid[0].split() if w.lower() not in STOPWORDS_ES and len(w) > 4]
        topic = " ".join(cands[:3]).capitalize() or "Segmento de clase"
    else:
        topic = "Segmento de clase"
    from collections import Counter
    freq = Counter(w for w in re.findall(r'\b[a-záéíóúüñ]{5,}\b', transcript.lower())
        if w not in STOPWORDS_ES)
    keywords = [w for w, _ in freq.most_common(6)]
    evidence = " ".join(transcript.split()[:18])
    return json.dumps({
        "topic": topic,
        "explanation": " ".join(bullets[:2]).replace("..", "."),
        "bullets": bullets,
        "example": valid[0] if valid else "Ver la grabación para ejemplos.",
        "formula": None,
        "keywords": keywords or ["concepto"],
        "timestamps_ms": [ts_start, ts_end],
        "source": {"slide_id": slide_id, "evidence": evidence[:120]},
    }, ensure_ascii=False, indent=2)
print(" validate_json_output, sanitize_output, heurística — nuevo schema")

SyntaxError: unterminated string literal (detected at line 48) (3690460037.py, line 48)

In [ ]:
def generate_pseudolabel(example: Dict, method: str) -> str:
    """Genera pseudo-label según el método configurado."""
    messages = example["messages"]
    slide_id = example["meta"]["slide_id"]
    timestamp_ms = example["meta"]["timestamp_ms"]
    try:
        if method == "llama_serve":
            return generate_with_ollama(messages)
        elif method == "heuristic":
            return generate_with_heuristics(messages, slide_id, timestamp_ms)
        else:
            raise ValueError(f"Método desconocido: {method}. Usa 'llama_serve' o 'heuristic'.")
    except Exception as e:
        print(f"\n Error en '{slide_id}': {e} → fallback a heurísticas")
        return generate_with_heuristics(messages, slide_id, timestamp_ms)
print(f" Generador configurado: {GENERATION_METHOD}")

 Generador configurado: llama_serve


---
## Procesar Train y Val

In [ ]:
# DEBUG: ver qué schema tiene el JSONL actual y qué devuelve Ollama
import random as _rnd
sample_ex = _rnd.choice(train_data)
# 1. ¿Qué system prompt tiene el JSONL?
sys_msg = next((m["content"] for m in sample_ex["messages"] if m["role"] == "system"), "")
has_new_schema = "topic" in sys_msg and "explanation" in sys_msg and "source" in sys_msg
has_old_schema = "title" in sys_msg and "slide_id" in sys_msg and "topic" not in sys_msg
print("" * 60)
print(" JSONL actual:")
print(f" Schema detectado : {'NUEVO ' if has_new_schema else 'VIEJO — necesitas re-ejecutar NB01' if has_old_schema else 'DESCONOCIDO'}")
print(f" System prompt : {sys_msg[:120]}...")
# 2. ¿Qué devuelve Ollama para 1 ejemplo?
if server_ok:
    print("\n Generando 1 ejemplo de prueba...")
    raw_output = _call_ollama(build_clean_messages(sample_ex["messages"]))
    print(f"\n Output crudo de Ollama (primeros 600 chars):")
    print("" * 60)
    print(raw_output[:600])
    print("" * 60)
    is_valid, parsed = validate_json_output(raw_output)
    print(f"\n validate_json_output → {'VÁLIDO' if is_valid else 'INVÁLIDO'}")
    if not is_valid:
        # Mostrar qué campo falla
        try:
            data = json.loads(extract_json(raw_output))
            missing = {'topic','explanation','bullets','example','formula','keywords','timestamps_ms','source'} - set(data.keys())
            print(f" Campos presentes : {list(data.keys())}")
            print(f" Campos faltantes : {missing}")
            bullets = data.get('bullets', [])
            print(f" bullets count : {len(bullets)} (necesita exactamente 5)")
            ts = data.get('timestamps_ms', [])
            print(f" timestamps_ms : {ts}")
        except Exception as e:
            print(f" No es JSON parseable: {e}")
else:
    print("\n Ollama no disponible — no se puede generar ejemplo de prueba")


 JSONL actual:
 Schema detectado : NUEVO 
 System prompt : Eres un experto en didáctica. Tu única tarea es transformar fragmentos de clase en apuntes estructurados.
SCHEMA OBLIGAT...

 Generando 1 ejemplo de prueba...

 Output crudo de Ollama (primeros 600 chars):

{
    "topic": "Aprendizaje automático en proyecciones",
    "explanation": "El aprendizaje automático se utiliza para prever comportamientos de compra basándose en datos recopilados de múltiples clientes.",
    "bullets": ["Recopilar datos de varios clientes", "Usar modelos predictivos", "Aplicar técnicas de machine learning", "Predecir tendencias de compra", "Optimizar inventario y ventas"],
    "example": "Un sistema que utiliza aprendizaje automático para prever las compras futuras basándose en datos históricos de clientes.",
    "formula": null,
    "keywords": ["aprendizaje automático", 


 validate_json_output → VÁLIDO


In [ ]:
def select_dry_sample(data: List[Dict], n: int, mode: str) -> List[Dict]:
    """Selecciona N ejemplos según el modo DRY_SAMPLE."""
    import random as _random
    if mode == "first":
        return data[:n]
    elif mode == "random":
        return [data[i] for i in sorted(_random.sample(range(len(data)), min(n, len(data))))]
    elif mode in ("shortest", "longest"):
        key = lambda ex: ex.get("meta", {}).get("chunk_words", 0)
        sorted_data = sorted(data, key=key, reverse=(mode == "longest"))
        return sorted_data[:n]
    return data[:n]
def run_generation(data: List[Dict], label: str, checkpoint_path: Path) -> List[Dict]:
    """
    Genera pseudo-labels con:
    - DRY_SAMPLE para selección de muestra
    - retries con "Fix your JSON" (en generate_with_ollama)
    - sanitize_output() inline
    - error handling: ejemplos fallidos marcados como error, no incluidos en final
    - checkpoint cada CHECKPOINT_EVERY ejemplos
    - reanuda desde checkpoint si existe
    """
    # Selección de subset
    if DRY_RUN:
        subset = select_dry_sample(data, DRY_RUN_N, DRY_SAMPLE)
        print(f" DRY_RUN={DRY_SAMPLE} — {len(subset)}/{len(data)} ejemplos")
    else:
        subset = data
    # Reanudar desde checkpoint
    results = []
    done_ids = set()
    if checkpoint_path.exists() and not DRY_RUN:
        results = load_jsonl(checkpoint_path)
        done_ids = {ex["id"] for ex in results}
        print(f" Checkpoint: {len(results)} ya procesados")
    pending = [ex for ex in subset if ex["id"] not in done_ids]
    print(f" {label}: {len(pending)} pendientes...")
    error_count = 0
    for i, example in enumerate(tqdm(pending, desc=label)):
        ex = deepcopy(example)
        slide_id = ex["meta"]["slide_id"]
        timestamp_ms = ex["meta"]["timestamp_ms"]
        user_content = next(
            (m["content"] for m in ex["messages"] if m["role"] == "user"), ""
        )
        # Generar
        try:
            if GENERATION_METHOD == "llama_serve":
                raw = generate_with_ollama(ex["messages"])
            else:
                raw = generate_with_heuristics(ex["messages"], slide_id, timestamp_ms)
        except Exception as e:
            # Fallo de red o timeout: fallback a heurísticas
            print(f"\n {slide_id}: {e} → heurística")
            raw = generate_with_heuristics(ex["messages"], slide_id, timestamp_ms)
        # Validar + sanitize inline
        is_valid, parsed = validate_json_output(raw)
        if is_valid:
            cleaned, _ = sanitize_output(parsed, user_content)
            final_content = json.dumps(cleaned, ensure_ascii=False)
        else:
            # Marcar como error: NO incluir en resultados finales
            error_count += 1
            ex["_error"] = {"msg": "JSON inválido tras retries", "raw_output": raw[:300]}
            # No añadir a results — continuar al siguiente
            continue
        for msg in reversed(ex["messages"]):
            if msg["role"] == "assistant":
                msg["content"] = final_content
                break
        results.append(ex)
        # Checkpoint
        if not DRY_RUN and (i + 1) % CHECKPOINT_EVERY == 0:
            save_jsonl(results, checkpoint_path)
            print(f" Checkpoint: {len(results)} guardados")
    if not DRY_RUN:
        save_jsonl(results, checkpoint_path)
    if error_count:
        print(f" {error_count} ejemplos descartados por JSON inválido tras {MAX_RETRIES} retries")
    return results
train_final = run_generation(train_data, "Train", CHECKPOINT_TRAIN)
print(f" Train: {len(train_final)} ejemplos válidos")

 Checkpoint: 676 ya procesados
 Train: 4 pendientes...


Train:   0%|          | 0/4 [00:00<?, ?it/s]

 Train: 680 ejemplos válidos


---
## Procesamiento de Validation Set

In [ ]:

val_final = run_generation(val_data, "Val", CHECKPOINT_VAL)
print(f" Val completado: {len(val_final)} ejemplos")


 Val: 125 pendientes...


Val:   0%|          | 0/125 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# DRY_RUN: mostrar outputs para validar calidad antes de continuar
if DRY_RUN:
    print("\n" + "="*70)
    print(f" DRY_RUN — mostrando {DRY_RUN_SHOW} outputs de muestra")
    print("="*70)
    sample = (train_final + val_final)[:DRY_RUN_SHOW]
    ok = 0
    contaminated = 0
    CONTAM_RE = re.compile(
        r'\b(SLIDE_OCR|SUBTITLES_OCR|TEACHER_AUDIO|TIME_RANGE|SLIDE_ID|'
        r'Genera los apuntes|frame_\d)\b|\b\d+ms\b', re.IGNORECASE
    )
    for i, ex in enumerate(sample, 1):
        user_msg = next(m["content"] for m in ex["messages"] if m["role"] == "user")
        asst_msg = next(m["content"] for m in ex["messages"] if m["role"] == "assistant")
        is_valid, parsed = validate_json_output(asst_msg)
        print(f"\n{''*70}")
        print(f"[{i}] {ex['id']} | has_ocr={ex['meta'].get('has_ocr','?')}")
        print(f"INPUT:\n{user_msg[:300]}")
        print(f"\nOUTPUT (válido={is_valid}):")
        if is_valid:
            ok += 1
            bullets = parsed.get("bullets", [])
            contam_bullets = [b for b in bullets if CONTAM_RE.search(b)]
            contaminated += len(contam_bullets)
            print(json.dumps(parsed, indent=2, ensure_ascii=False))
            if contam_bullets:
                print(f" Bullets contaminados: {contam_bullets}")
        else:
            print(asst_msg[:400])
    total = len(train_final) + len(val_final)
    print(f"\n{'='*70}")
    print(f" DRY_RUN resumen: {ok}/{len(sample)} JSON válidos | {contaminated} bullets contaminados")
    print(f" Si hay contaminación → ajusta build_clean_messages() o el system prompt")
    print(f" Si todo OK → pon DRY_RUN=False y relanza")
    print("="*70)

---
## Validación de JSON

In [ ]:
# Validación sobre los datos generados
# (validate_json_output y extract_json ya definidos en cell-10)
def count_valid(dataset):
    valid, empty = 0, 0
    for ex in dataset:
        msg = next((m for m in ex['messages'] if m['role'] == 'assistant'), None)
        if not msg or not msg['content'].strip():
            empty += 1
            continue
        if validate_json_output(msg['content'])[0]:
            valid += 1
    return valid, empty
train_valid, train_empty = count_valid(train_final)
val_valid, val_empty = count_valid(val_final)
print(f" Validación JSON:")
print(f" Train: {train_valid}/{len(train_final)} válidos | {train_empty} vacíos")
print(f" Val: {val_valid}/{len(val_final)} válidos | {val_empty} vacíos")

---
## Validación y Estadísticas

In [ ]:
def compute_quality_stats(dataset: List[Dict]) -> Dict:
    """Calcula estadísticas de calidad del dataset."""
    valid_count = 0
    bullet_counts = []
    keyword_counts = []
    output_lengths = []
    for ex in dataset:
        assistant_msg = [m for m in ex['messages'] if m['role'] == 'assistant'][0]
        content = assistant_msg['content']
        is_valid, data = validate_json_output(content)
        if is_valid:
            valid_count += 1
            bullet_counts.append(len(data['bullets']))
            keyword_counts.append(len(data['keywords']))
            output_lengths.append(len(content))
    return {
        "total_examples": len(dataset),
        "valid_json": valid_count,
        "valid_percentage": valid_count / len(dataset) * 100,
        "avg_bullets": sum(bullet_counts) / len(bullet_counts) if bullet_counts else 0,
        "avg_keywords": sum(keyword_counts) / len(keyword_counts) if keyword_counts else 0,
        "avg_output_length": sum(output_lengths) / len(output_lengths)
    }
train_stats = compute_quality_stats(train_final)
val_stats = compute_quality_stats(val_final)
print("\n ESTADÍSTICAS DE CALIDAD")
print("\nTrain:")
for k, v in train_stats.items():
    print(f" {k}: {v:.2f}" if isinstance(v, float) else f" {k}: {v}")
print("\nValidation:")
for k, v in val_stats.items():
    print(f" {k}: {v:.2f}" if isinstance(v, float) else f" {k}: {v}")

---
## Guardar Datasets Finales

In [ ]:
def save_jsonl(data: List[Dict], filepath: Path):
    """Guarda lista de dicts como JSONL."""
    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
# Guardar datasets
save_jsonl(train_final, FINAL_TRAIN_PATH)
save_jsonl(val_final, FINAL_VAL_PATH)
# Guardar stats
stats = {
    "generation_method": GENERATION_METHOD,
    "train": train_stats,
    "validation": val_stats
}
with open(STATS_PATH, 'w') as f:
    json.dump(stats, f, indent=2)
print(f"\n Archivos guardados:")
print(f" Train: {FINAL_TRAIN_PATH} ({FINAL_TRAIN_PATH.stat().st_size / 1024:.1f} KB)")
print(f" Val: {FINAL_VAL_PATH} ({FINAL_VAL_PATH.stat().st_size / 1024:.1f} KB)")
print(f" Stats: {STATS_PATH}")

---
## Inspección de Ejemplos Finales

In [ ]:
print("\n" + "="*80)
print(" EJEMPLOS CON PSEUDO-LABELS")
print("="*80)
for i, example in enumerate(train_final[:3], 1):
    print(f"\n{'='*80}")
    print(f"Ejemplo {i}: {example['id']}")
    print(f"{'='*80}")
    # Mostrar solo user y assistant
    for msg in example['messages']:
        if msg['role'] == 'user':
            print(f"\n[USER]")
            print(msg['content'][:200] + "...")
        elif msg['role'] == 'assistant':
            print(f"\n[ASSISTANT]")
            # Pretty print JSON si es válido
            is_valid, data = validate_json_output(msg['content'])
            if is_valid:
                print(json.dumps(data, indent=2, ensure_ascii=False))
            else:
                print(msg['content'][:300] + "...")
    print(f"\n{'-'*80}")

---
## Resumen Final

In [ ]:
print("\n" + "="*80)
print(" NOTEBOOK 03 COMPLETADO")
print("="*80)
print(f"\n Archivos generados:")
print(f" - {FINAL_TRAIN_PATH}")
print(f" - {FINAL_VAL_PATH}")
print(f" - {STATS_PATH}")
print(f"\n Calidad:")
print(f" - Train: {train_stats['valid_json']}/{train_stats['total_examples']} válidos ({train_stats['valid_percentage']:.1f}%)")
print(f" - Val: {val_stats['valid_json']}/{val_stats['total_examples']} válidos ({val_stats['valid_percentage']:.1f}%)")
print(f" - Método: {GENERATION_METHOD}")
print(f"\n Próximos pasos:")
print(f" - Notebook 04: RAG Data Preparation (preparar chunks para indexación)")
print(f" - Notebook 05: RAG Indexing (construir índice FAISS+BM25)")
print(f" - notebooks_colab/: Fine-tuning con QLoRA (requiere GPU)")
print("="*80)

---
## Troubleshooting

### JSON inválido (< 95%)
- Bajar `temperature` en Ollama (actualmente 0.3)
- Aumentar `MAX_RETRIES` para más intentos de corrección
- Revisar el system prompt en `raw_train.jsonl`

### Ollama no responde
- Verificar que está corriendo: `ollama serve`
- Verificar modelo disponible: `ollama list`
- Si falla, usar `GENERATION_METHOD = "heuristic"` como fallback

### Bullets contaminados con metadatos
- Ajustar `build_clean_messages()` para filtrar mejor el input
- `sanitize_output()` limpia la mayoría de contaminación automáticamente